## Libraries

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import random
import tensorflow as tf
import tensorflow_io as tfio
from tensorflow import keras
from keras import layers

print(f"Tensorflow {tf.__version__}")
print(f"Keras {keras.__version__}")

Tensorflow 2.16.2
Keras 3.13.2


## Dataset

In [9]:
data_dir = "/Volumes/Dafna/venv-tf/share/librispeech" ## CHANGE THIS
train_name = "train-clean-100"
test_name = "test-clean"
dev_name = "dev-clean"

### Download

The following cell downloads the sets defined above (in `train_name`, `test_name` and `dev_name`) into `data_dir`,
from <https://www.openslr.org/12>.

The downloaded archive is structured like `LibriSpeech/<dataset>/<speaker>/<chapter>/<file>.flac`. This function also takes care to reorganize that
into `<dataset>/<speaker>/<file>.flac` (we don't care about chapter structure).

In [11]:
import shutil
for requirement in ["curl", "tar", "grep"]:
    if not shutil.which(requirement):
        raise f"Dependency {requirement} not found!"

def ensure_set(name: str):
    basepath = Path(data_dir)
    set_dir = basepath.joinpath(name)
    tar = basepath.joinpath(name + ".tar.gz")
    basepath.mkdir(exist_ok=True)

    if set_dir.exists():
        print(f"Set {name} already found")
        return

    if not tar.exists():
        print(f"Downloading Librispeech {name}...")
        url = "https://openslr.trmal.net/resources/12/" + name + ".tar.gz"
        !curl -Lo {tar.absolute()} {url}

    print(f"Extracting Librispeech {name}")
    set_dir.mkdir()
    speakers = !tar -tf {tar.absolute()} | grep '.*/dev-clean/[[:digit:]]*/$'
    for speaker in speakers:
        sp_path = Path(speaker)
        path = set_dir.joinpath(sp_path.name)
        path.mkdir()
        !tar -C {path.absolute()} --strip-components 4 -xzf {tar.absolute()} {sp_path}'/**/*.flac'
    %rm {tar.absolute()}

print("Downloading dataset (Librispeech)")
ensure_set(train_name)
ensure_set(dev_name)
ensure_set(test_name)
print("Done!")

Set train-clean-100 already found
Set dev-clean already found
Set test-clean already found
Done!


### Setup

In [ ]:
n_speakers = 32
n_utterances = 4
batch_size = n_speakers * n_utterances
sample_rate = 16000

# librispeech_train = tfds.load(
#   "librispeech", split="train_clean100[:20%]",
#   builder_kwargs={"config": "lazy_decode"},
#   data_dir="/Volumes/Dafna/venv-tf/share/tfds-data"
# )

# samples_by_speaker = {}
# for ex in tfds.as_numpy(librispeech_train):
#   sid = int(ex["speaker_id"])
#   if sid not in samples_by_speaker: samples_by_speaker[sid] = []
#   samples_by_speaker[sid].append(ex["audio"])

# def batches():
#   all_ids = list(samples_by_speaker.keys())
#   while True:
#     sids = random.sample(all_ids, n_speakers)
#     waves = []
#     labels = []
#     for sid in sids:
#       exs = samples_by_speaker[sid]
#       utterances = random.choices(exs, k=n_utterances) if len(exs) < n_utterances else random.sample(exs, n_utterances)
#       for u in utterances:
#         samp = tf.convert_to_tensor(u[0], dtype=tf.float32)
#         waves.append(samp)
#         labels.append(sid)
#     yield waves, labels

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6091M  100 6091M    0     0  38.2M      0  0:02:39  0:02:39 --:--:-- 32.0M11  708M    0     0  39.1M      0  0:02:35  0:00:18  0:02:17 43.2M 0     0  37.9M      0  0:02:40  0:00:33  0:02:07 41.9M0M    0     0  37.6M      0  0:02:41  0:00:47  0:01:54 33.2M     0  37.4M      0  0:02:42  0:00:56  0:01:46 34.5M20M    0     0  37.0M      0  0:02:44  0:00:57  0:01:47 30.0M46M    0     0  36.9M      0  0:02:44  0:00:58  0:01:46 28.3M 0     0  37.5M      0  0:02:42  0:01:08  0:01:34 36.7M0     0  37.6M      0  0:02:41  0:01:09  0:01:32 36.5M   0     0  38.2M      0  0:02:39  0:01:15  0:01:24 44.6M49 3009M    0     0  38.0M      0  0:02:40  0:01:19  0:01:21 36.4M 0     0  38.1M      0  0:02:39  0:01:21  0:01:18 39.0M2 3185M    0     0  38.3M      0  0:02:38  0:01:23  0:01:15 44.3M7M    0     0  38.0M      0  0:02:39  0:01:26  0:01:13 3

### Raw Audio

In [ ]:
clip_ms = 1000
input_samples = int((sample_rate / 1000) * clip_ms)

### Spectrograms (DFT)

In [ ]:
spectrogram_shape=(256, 512, 1)

## Models

### Helper Functions

### General Training Pipeline

In [ ]:
def pairwise_distances(embeddings):
  square = tf.reduce_sum(tf.square(embeddings), axis=1, keepdims=True)
  dist = square - 2.0 * tf.matmul(embeddings, embeddings, transpose_b=True) + tf.transpose(square)
  return tf.maximum(dist, 0.0)

def triplet_loss(y, embeddings, margin=0.2):
  dist = pairwise_distances(embeddings)

  # mask for positives/negatives
  y_eq = tf.equal(tf.expand_dims(y, 1), tf.expand_dims(y, 0))
  mask_pos = tf.cast(y_eq, tf.float32) - tf.eye(tf.shape(y)[0])
  mask_neg = 1.0 - tf.cast(y_eq, tf.float32)

  # hardest positive: max distance among positives
  hardest_pos = tf.reduce_max(dist * mask_pos, axis=1)
  # hardest negative: min distance among negatives (add large value where no neg)
  max_val = tf.reduce_max(dist) + 1.0
  pdist_neg = dist + max_val * (1.0 - mask_neg)
  hardest_neg = tf.reduce_min(pdist_neg, axis=1)

  tloss = tf.maximum(0.0, hardest_pos - hardest_neg + margin)
  return tf.reduce_mean(tloss)

class TripletTrainer(keras.Model):
  def __init__(self, embedded, margin=0.2):
    super().__init__()
    self.embedded = embedded
    self.margin = margin

  def compile(self, optimizer, *args, **kwargs):
    super().compile(*args, **kwargs)
    self.optimizer = optimizer

  @tf.function
  def train_step(self, data):
    x, y = data
    with tf.GradientTape() as tape:
      embeddings = self.embedded(x, training = True)
      loss = triplet_loss(y, embeddings, margin=self.margin)
      grads = tape.gradient(loss, self.embedded.trainable_variables)

    self.optimizer.apply_gradients(zip(grads, self.embedded.trainable_variables))
    return {"loss": loss}

  @tf.function
  def test_step(self, data):
    x, y = data
    embeddings = self.embedded(x, training = False)
    loss = triplet_loss(y, embeddings, margin=self.margin)
    return {"loss": loss}

### 1D CNN on Raw Audio

In [ ]:
audio_cnn = keras.Sequential([
  layers.Input(shape=(input_samples, 1)),
  layers.Conv1D(64, sample_rate // 10, strides=sample_rate // 2000, activation="relu"),
  layers.Conv1D(128, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(128, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(256, 3, activation="relu", padding="same"),
  layers.Conv1D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(256, 3, activation="relu", padding="same"),
  layers.Conv1D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),

  layers.Flatten(),
  layers.Dense(4096, activation="relu"),
  layers.Dropout(0.2),
  layers.Dense(2048, activation="relu"),
  layers.BatchNormalization(),
], name="1D CNN on Raw Audio")

audio_cnn.summary()

Model: "1D CNN on Raw Audio"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_7 (Conv1D)               │ (None, 1801, 64)       │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 1801, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1801, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 900, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 900, 128)       │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 900, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 450, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 450, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 450, 256)       │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 450, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 225, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_12 (Conv1D)              │ (None, 225, 256)       │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 225, 256)       │       196,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 225, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 112, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 28672)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4096)           │   117,444,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,712,128 (483.37 MB)

 Trainable params: 126,706,496 (483.35 MB)

 Non-trainable params: 5,632 (22.00 KB)

### 1D CNN on Spectrogram

In [ ]:
spect_1d = keras.Sequential([
  layers.Input(shape=spectrogram_shape),
  layers.Normalization(),
  # The width dimension represents time, and the height represents frequencies.
  # We treat the frequencies as "features", so we can convolve over them.
  layers.Reshape((spectrogram_shape[0], spectrogram_shape[1])),
  layers.Conv1D(spectrogram_shape[1], 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.Conv1D(512, 3, activation="relu", padding="same"),
  layers.Conv1D(1024, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(2048, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(4096, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),
  layers.Conv1D(4096, 3, activation="relu", padding="same"),
  layers.Conv1D(8192, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling1D(),

  layers.Flatten(),
  layers.Dense(4096, activation="relu"),
  layers.Dropout(0.2),
  layers.Dense(2048, activation="relu"),
  layers.BatchNormalization(),
], name="1D CNN on Spectrogram")

spect_1d.summary()

Model: "1D CNN on Spectrogram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (None, 256, 512, 1)    │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 256, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_14 (Conv1D)              │ (None, 256, 512)       │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 256, 512)       │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 128, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_15 (Conv1D)              │ (None, 128, 256)       │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 128, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_16 (Conv1D)              │ (None, 128, 512)       │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 128, 1024)      │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128, 1024)      │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 64, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 64, 2048)       │     6,293,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 64, 2048)       │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 32, 2048)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 32, 4096)       │    25,169,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 32, 4096)       │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 16, 4096)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_20 (Conv1D)              │ (None, 16, 4096)       │    50,335,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_21 (Conv1D)              │ (None, 16, 8192)       │   100,671,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 16, 8192)       │        32,768 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 8, 8192)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 65536)          │             

 Total params: 462,521,603 (1.72 GB)

 Trainable params: 462,485,248 (1.72 GB)

 Non-trainable params: 36,355 (142.02 KB)

### 2D CNN on Spectrogram (Mobilenet)

In [ ]:
mobilenet = keras.applications.MobileNetV2(
  input_shape=spectrogram_shape,
  include_top=False,
  weights=None
)

mobilenet.summary()

Model: "mobilenetv2_1.00_256"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_73      │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 128, 256,  │        288 │ input_layer_73[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 128, 256,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 128, 256,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 256,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 256,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 256,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 256,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 256,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 128, 256,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 128, 256,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 128, 256,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 129, 257,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 64, 128,   │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 128,   │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 64, 128,   │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 64, 128,   │      2,304 │ block_1_depthwis

 Total params: 2,257,408 (8.61 MB)

 Trainable params: 2,223,296 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

### 2D Bespoke CNN on Spectrogram

In [ ]:
spect_2d = keras.Sequential([
  layers.Input(shape=spectrogram_shape),
  layers.Normalization(),
  layers.Conv2D(32, 3, activation="relu", padding="same"),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),
  layers.Conv2D(128, 3, activation="relu", padding="same"),
  layers.Conv2D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),
  layers.Conv2D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),
  layers.Conv2D(256, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),
  layers.Conv2D(512, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),
  layers.Conv2D(1024, 3, activation="relu", padding="same"),
  layers.BatchNormalization(),
  layers.MaxPooling2D(),

  layers.Flatten(),
  layers.Dense(4096, activation="relu"),
  layers.Dropout(0.2),
  layers.Dense(2048, activation="relu"),
  layers.BatchNormalization(),
], name="2D CNN on Spectrogram")

spect_2d.summary()

Model: "2D CNN on Spectrogram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_1 (Normalization) │ (None, 256, 512, 1)    │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 512, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 256, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 128, 256, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 128, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 128, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 128, 256)   │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 64, 128, 256)   │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 64, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 32, 64, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 16, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 16, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 8, 16, 512)     │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 8, 16, 512)     │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 4, 8, 1024)     │     4,719,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 4, 8, 1024)     │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 2, 4, 1024)     │             

 Total params: 49,434,627 (188.58 MB)

 Trainable params: 49,425,792 (188.54 MB)

 Non-trainable params: 8,835 (34.52 KB)